In [3]:
import os
import sys
sys.path.append(os.path.abspath(".."))


In [38]:

import importlib
import src.data_loader
import src.preprocessing
import src.collab_model
import src.content_model
import src.hybrid
importlib.reload(src.content_model)
importlib.reload(src.collab_model)
importlib.reload(src.data_loader)
importlib.reload(src.preprocessing)
importlib.reload(src.hybrid)

<module 'src.hybrid' from 'c:\\Users\\saidmzz\\Desktop\\movie_recommender\\src\\hybrid.py'>

In [5]:
from src.data_loader import load_ratings, load_movies, load_tags

ratings = load_ratings()
print(ratings.shape)
print(ratings.head())

(20000263, 4)
   userId  movieId  rating            timestamp
0       1        2     3.5  2005-04-02 23:53:47
1       1       29     3.5  2005-04-02 23:31:16
2       1       32     3.5  2005-04-02 23:33:39
3       1       47     3.5  2005-04-02 23:32:07
4       1       50     3.5  2005-04-02 23:29:40


In [33]:
from src.preprocessing import build_movie_content, convert_to_implicit

movies = load_movies()
tags = load_tags()

movie_content = build_movie_content(movies, tags)
print(movie_content.head())

   movieId                                            content
0        1  Toy Story (1995) Adventure Animation Children ...
1        2  Jumanji (1995) Adventure Children Fantasy time...
2        3  Grumpier Old Men (1995) Comedy Romance old peo...
3        4  Waiting to Exhale (1995) Comedy Drama Romance ...
4        5  Father of the Bride Part II (1995) Comedy Dian...


In [34]:
from src.collab_model import build_interaction_matrix, train_als

ratings = load_ratings().sample(500000)
ratings = convert_to_implicit(ratings)

matrix, user_mapping, movie_mapping = build_interaction_matrix(ratings)
model = train_als(matrix)

print("ALS trained successfully")

100%|██████████| 15/15 [00:02<00:00,  5.02it/s]

ALS trained successfully


In [35]:
from src.content_model import train_content_embeddings
# subset = movie_content.head(100)

# # generate embeddings
# embeddings = train_content_embeddings(subset)

# print("Shape:", embeddings.shape)

In [ ]:
# from sklearn.metrics.pairwise import cosine_similarity
# import numpy as np

# # Compute similarity between first 5 movies
# sim_matrix = cosine_similarity(embeddings[:5])

# print(np.round(sim_matrix, 2))

[[1.   0.52 0.42 0.25 0.18]
 [0.52 1.   0.24 0.31 0.23]
 [0.42 0.24 1.   0.43 0.4 ]
 [0.25 0.31 0.43 1.   0.3 ]
 [0.18 0.23 0.4  0.3  1.  ]]


In [ ]:
# def find_similar_movies(index, embeddings, movie_content, top_k=5):
#     sims = cosine_similarity(
#         [embeddings[index]],
#         embeddings
#     )[0]
    
#     top_indices = sims.argsort()[-top_k-1:][::-1]
    
#     for i in top_indices:
#         print(movie_content.iloc[i]["content"][:80])

# find_similar_movies(0, embeddings, subset, top_k=5)

Toy Story (1995) Adventure Animation Children Comedy Fantasy Watched computer an
Pocahontas (1995) Animation Children Drama Musical Romance Disney animated featu
Babe (1995) Children Drama animal:pig barnyard animals Animal movie children ove
Jumanji (1995) Adventure Children Fantasy time travel adapted from:book board ga
City of Lost Children, The (Cité des enfants perdus, La) (1995) Adventure Drama 
Kids of the Round Table (1995) Adventure Children Fantasy 


In [36]:
content_embeddings = train_content_embeddings(movie_content)

print("Shape:", content_embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 651.81it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 853/853 [02:45<00:00,  5.14it/s]

Shape: (27278, 384)


In [39]:
from src.hybrid import recommend_hybrid
user_id = list(user_mapping.values())[0]

recs = recommend_hybrid(
    user_id,
    model,
    matrix,
    user_mapping,
    movie_mapping,
    content_embeddings,
    movie_content,
    top_k=10
)

print("Recommendations:", recs)

Recommendations: [4993, 2021, 4886, 4979, 47610, 2750, 1148, 3994, 6296, 1222]
